In [1]:
import sys
sys.path.append("..")

In [3]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [4]:
def append_result(d, objective, loss, cost, m1_validity, wc_validity, m1_expectation, wc_expectation):
    d['cost'].append(cost)
    d['m1_validity'].append(m1_validity)
    d['wc_validity'].append(wc_validity)
    d['m1_probability'].append(m1_expectation)
    d['wc_probability'].append(wc_expectation)
    d['loss'].append(loss)
    d['J'].append(objective) 
    
def get_result(d, algorithm, seed, alpha, lamb, theta_0, theta_r):
    result = {
        'algorithm': algorithm, 
        'seed': seed,
        'alpha': alpha,
        'lambda': lamb,
        'theta_0': theta_0,
        'theta_r': theta_r
        }
    
    for key in d.keys():
        result[key] = np.mean(d[key])
    return result

In [32]:
def get_theta_adv_search(X_0, X_r, theta_0, alpha, lamb):
    
    theta_adv = deepcopy(theta_0)
    # X_0 = torch.tensor(np.stack(X_0)).float()
    # X_r = torch.tensor(np.stack(X_r)).float()
    
    for i in range(theta_0.shape[0]):
        theta_r_min = deepcopy(theta_0)
        theta_r_max = deepcopy(theta_0)
        
        theta_r_min[i] -= alpha
        theta_r_max[i] += alpha
        weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
        weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
        J_min, J_max = [], []
        for xi in range(len(X_r)):
            x_0 = X_0[xi]
            x_r = X_r[xi]
            J = RecourseCost(x_0, lamb)
            j_min = J.eval(x_r, weights_r_min, bias_r_min)
            j_max = J.eval(x_r, weights_r_max, bias_r_max)
            J_min.append(j_min)
            J_max.append(j_max)
        
        
        if np.mean(J_min) > np.mean(J_max):
            theta_adv[i] -= alpha
        else:
            theta_adv[i] += alpha

    return theta_adv

In [33]:
# def get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb):
#     i_max = 0
#     alpha_max = 0
#     val_max = -np.inf

#     for i in range(theta_0.shape[0]):
#     # for i in range(X_0.shape[1]):
#         theta_r_min = deepcopy(theta_0)
#         theta_r_max = deepcopy(theta_0)
        
#         theta_r_min[i] -= alpha
#         theta_r_max[i] += alpha
#         weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
#         weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
#         J_min, J_max = [], []
#         for xi in range(len(X_r)):
#             x_0 = X_0[xi]
#             x_r = X_r[xi]
#             J = RecourseCost(x_0, lamb)
#             j_min = J.eval(x_r, weights_r_min, bias_r_min)
#             j_max = J.eval(x_r, weights_r_max, bias_r_max)
#             J_min.append(j_min)
#             J_max.append(j_max)
        
#         if np.mean(J_min) > np.mean(J_max):
#             if np.mean(J_min) > val_max:
#                 i_max = i
#                 alpha_max = -alpha
#                 val_max = np.mean(J_min).item()
#         else:
#             if np.mean(J_max) > val_max:
#                 i_max = i
#                 alpha_max = alpha
#                 val_max = np.mean(J_max).item()
    
#     theta_adv = deepcopy(theta_0)
#     theta_adv[i_max] += alpha_max
#     return theta_adv

def generateThetas(theta0 : np.ndarray, alpha):
        # theta0 has bias
        thetas = theta0.copy()
        if alpha == 0:
            return np.array([thetas])
        
        thetas = np.repeat(thetas.reshape(1, theta0.size), (theta0.size * 2) - 1, axis=0)
        thetas_i = 0

        for i in range(theta0.size):
            if i == theta0.size - 1:
                thetas[thetas_i][i] -= alpha
                thetas_i += 1
                break

            thetas[thetas_i][i] += alpha
            thetas_i += 1
            thetas[thetas_i][i] -= alpha
            thetas_i += 1

        return thetas

def get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb):
    thetas = generateThetas(theta_0, alpha)
    if alpha == 0:
        return thetas[0].copy()
    
    Js = np.empty((X_0.shape[0], thetas.shape[0]))
    for i in range(X_0.shape[0]):
         J = RecourseCost(X_0[i], lamb)
         for j, theta in enumerate(thetas):  
            Js[i, j] = J.eval(X_r[i], theta[:-1], np.array([theta[-1]]))
    
    Js_sum = Js.sum(axis=0) 
    Js_sum_maxI = np.argmax(Js_sum)
    theta_adv = thetas[Js_sum_maxI]

    return theta_adv

In [34]:
def evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    weights_0, bias_0 = theta_0[:-1], theta_0[[-1]]

    if theta_adv_method=='L-1':
        theta_adv = get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb)
    else:
        theta_adv = get_theta_adv_search(X_0, X_r, theta_0, alpha, lamb)
    
    weights_adv, bias_adv = theta_adv[:-1], theta_adv[[-1]]
        
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    clf = LR()
    clf.train(X_0, Y_0)
    clf.model.coef_ = weights_0.reshape(1,-1)
    clf.model.intercept_ = bias_0
    
    clf_adv = deepcopy(clf)
    clf_adv.model.coef_ = weights_adv.reshape(1,-1)
    clf_adv.model.intercept_ = bias_adv

    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        J = RecourseCost(x_0, lamb)
        
        bce_loss, cost, price = J.eval(x_r, weights_adv, bias_adv, True)
        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, theta_0, theta_adv)

In [ ]:
def calThetaAdv_l1(xP: np.ndarray, theta0: np.ndarray, alpha):
    # xP has bias
    thetaP = theta0.copy()
    i = np.argmax(np.abs(xP))
    thetaP[i] -= (alpha * np.sign(xP[i]))

    return thetaP

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.concat((weights_adv, bias_adv))

def calTheta(xP: np.array, weights: np.array, bias: np.array, alpha: float, methods: str):
    if methods == "L-inf":
        thetaP = calThetaAdv_linf(xP, weights, bias, alpha)
    else:
        thetaP = calThetaAdv_l1(np.hstack((xP, np.array([1]))), np.hstack((weights, bias)), alpha)

    return thetaP[:-1], np.array([thetaP[-1]])

In [36]:
def evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    
    weights_0, bias_0 = theta_0[:-1], theta_0[[-1]]
    # Don't need this block of code below
    if theta_adv_method=='L-1':
        theta_adv = get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb)
    else:
        theta_adv = get_theta_adv_search(X_0, X_r, theta_0, alpha, lamb)
    # weights_adv, bias_adv = theta_adv[:-1], theta_adv[[-1]]
        
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    
    clf = LR()
    clf.train(X_0, Y_0)
    clf.model.coef_ = weights_0.reshape(1,-1)
    clf.model.intercept_ = bias_0    
    clf_adv = deepcopy(clf)


    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]

        weights_adv, bias_adv = calTheta(x_r, weights_0, bias_0, alpha, theta_adv_method)
        clf_adv.model.coef_ = weights_adv.reshape(1,-1)
        clf_adv.model.intercept_ = bias_adv

        J = RecourseCost(x_0, lamb)
        
        bce_loss, cost, price = J.eval(x_r, weights_adv, bias_adv, True)
        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, theta_0, theta_adv)

In [285]:
def evaluate_performance_largestAlpha(X_0, X_r, theta_0, alpha, maxAlpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    alpha = maxAlpha

    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    weights_0, bias_0 = theta_0[:-1], theta_0[[-1]]

    if theta_adv_method=='L-1':
        theta_adv = get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb)
    else:
        theta_adv = get_theta_adv_search(X_0, X_r, theta_0, alpha, lamb)
    
    weights_adv, bias_adv = theta_adv[:-1], theta_adv[[-1]]
        
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    clf = LR()
    clf.train(X_0, Y_0)
    clf.model.coef_ = weights_0.reshape(1,-1)
    clf.model.intercept_ = bias_0
    
    clf_adv = deepcopy(clf)
    clf_adv.model.coef_ = weights_adv.reshape(1,-1)
    clf_adv.model.intercept_ = bias_adv

    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        J = RecourseCost(x_0, lamb)
        
        bce_loss, cost, price = J.eval(x_r, weights_adv, bias_adv, True)
        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        # append_result(results, price, bce_loss, cost, m1_validity, wc_probability, m1_probability, wc_probability)

    # print(theta_adv)    
    return get_result(results, algorithm, seed, alpha, lamb, theta_0, theta_adv)

In [307]:
alphas = [0.02]
lambdas = [0.1]
# alphas = np.arange(0.02, 0.52, 0.02).round(2)
# lambdas = [0.05, 0.1, 0.2, 0.3]
maxAlpha = 0.5

params = {}
# 'lr', 'nn'
params['base_model'] = 'lr'
# 'synthetic', 'german', 'sba'
params['data'] = 'synthetic'
params['seeds'] = range(5)
# TODO: add your method here, the method name should match the name in filepath
# params['algorithms'] = ['Alg1', 'L1PSD', 'ROARLInf', 'ROARL1']
params['algorithms'] = ['Alg1', 'ROAR']
# 'one', 'many'
params['adv_method'] = 'alpha'

results = {
    'algorithm': [],
    'seed': [],
    'alpha': [],
    'lambda': [],
    'Cost': [],
    'Current Validity': [],
    'Worst Case Validity': [],
    'BCE Loss': [],
    'J': []
}

for algorithm in params['algorithms']:
    for seed in params['seeds']:
        for v_alpha in alphas:
            for v_lamb in lambdas:
                # data = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")
                data = pd.read_pickle(f"../results/recourse-2025_08/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")
                alpha = data["alpha"].unique().item()
                lamb = data["lambda"].unique().item()
                theta_0 = data["theta_0"].iloc[0]
                weights_0, bias_0, = theta_0[:-1], theta_0[[-1]]
                X_0 = np.stack(data["x_0"])
                X_r = np.stack(data["x_r"])
                if params['adv_method'] == 'one':
                    res = evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                elif params['adv_method'] == 'alpha':
                    res = evaluate_performance_largestAlpha(X_0, X_r, theta_0, alpha, maxAlpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                else:
                    res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                results['algorithm'].append(algorithm)
                results['seed'].append(seed)
                results['alpha'].append(alpha)
                results['lambda'].append(lamb)
                results['Cost'].append(res['cost'])
                results['Current Validity'].append(res['m1_probability'])
                results['Worst Case Validity'].append(res['wc_probability'])
                results['BCE Loss'].append(res['loss'])
                results['J'].append(res['J'])

df_results = pd.DataFrame(results)

[Roar] [ seed=4 ] [ α=0.5 ] [ λ=0.1 ]: 100%|██████████| 105/105 [00:00<00:00, 2652.67it/s]


In [287]:
# df_graph = df_results.groupby(["alpha", "lambda", "algorithm"]).mean().reset_index()
# mask = df_graph['algorithm'] == 'Alg1'
# df_graph.loc[mask,'algorithm'] = 'LInf'
# df_graph['algorithm_lamb'] = df_graph[['algorithm', 'lambda']].apply(lambda row: row['algorithm'] + '(Lamb = ' + str(row['lambda']) + ')' , axis=1)


# # df_graph
# # df_graph[(df_graph['algorithm'] == "LInf") | (df_graph['algorithm'] == "L1PSD")]
# df_graph[(df_graph['algorithm'] == "LInf")]



print(f'{params["data"]}  |  {params["base_model"].upper()}')
df_results_avg = df_results.groupby(['algorithm', 'lambda'], as_index=False).mean(True)
df_results_im = df_results_avg.copy()
df_results_im[['Cost', 'Current Validity', 'Worst Case Validity', 'J']] = df_results_im[['Cost', 'Current Validity', 'Worst Case Validity', 'J']].round(2).astype(str) + '±' + df_results.groupby(['algorithm', 'lambda'], as_index=False).std(numeric_only=True)[['Cost', 'Current Validity', 'Worst Case Validity', 'J']].round(2).astype(str)

df_results_im

sba  |  LR


,algorithm,lambda,seed,alpha,Cost,Current Validity,Worst Case Validity,BCE Loss,J
0,Alg1,0.05,2.0,0.26,3.91±0.78,1.0±0.0,0.72±0.25,0.485492,0.68±0.5
1,Alg1,0.10,2.0,0.26,3.67±0.77,1.0±0.01,0.65±0.27,0.682830,1.05±0.62
2,Alg1,0.20,2.0,0.26,3.43±0.76,0.99±0.01,0.57±0.28,0.935590,1.62±0.72
3,Alg1,0.30,2.0,0.26,3.29±0.75,0.99±0.02,0.52±0.28,1.119803,2.11±0.76
4,ROAR,0.05,2.0,0.26,10.56±3.08,0.97±0.02,0.63±0.26,0.982179,1.51±0.73
5,ROAR,0.10,2.0,0.26,6.44±1.82,0.97±0.02,0.58±0.3,1.233490,1.88±0.83
6,ROAR,0.20,2.0,0.26,4.71±1.52,0.96±0.05,0.5±0.31,1.690903,2.63±1.04
7,ROAR,0.30,2.0,0.26,4.05±1.37,0.94±0.09,0.46±0.31,2.030124,3.25±1.16


In [288]:
# # custom_colors = {
# #     "LInf(Lamb = 0.1)": "#33FFFF",
# #     "L1PSD(Lamb = 0.1)": "#FF3333",
# #     "ROARL1(Lamb = 0.1)": "#33FF33",
# #     "ROARLInf(Lamb = 0.1)": "#FF33FF",
# #     "LInf(Lamb = 0.3)": "#33FFFF",
# #     "L1PSD(Lamb = 0.3)": "#FF3333",
# #     "ROARL1(Lamb = 0.3)": "#33FF33",
# #     "ROARLInf(Lamb = 0.3)": "#FF33FF"
# # }

# custom_colors = {
#     "LInf(Lamb = 0.1)": "#33FFFF",
#     "L1PSD(Lamb = 0.1)": "#FF3333",
#     "ROARL1(Lamb = 0.1)": "#33FF33",
#     "ROARLInf(Lamb = 0.1)": "#FF33FF",
#     "LInf(Lamb = 0.2)": "#33FFFF",
#     "L1PSD(Lamb = 0.2)": "#FF3333",
#     "ROARL1(Lamb = 0.2)": "#33FF33",
#     "ROARLInf(Lamb = 0.2)": "#FF33FF"
# }

# fig = px.line(df_graph, 
#            x="Cost", y="Worst Case Validity", 
#            color="algorithm_lamb",
#            hover_data=["alpha", "lambda"],
#            title=f"{params['base_model']}_{params['data']}_{params['adv_method']}Adv",
#            markers=True,
#            color_discrete_map=custom_colors,
#            facet_col="lambda") 
# fig
    

In [289]:
df_results_avg = df_results.groupby(['algorithm', 'lambda', 'alpha'], as_index=False).mean()
df_results_avg

,algorithm,lambda,alpha,seed,Cost,Current Validity,Worst Case Validity,BCE Loss,J
0,Alg1,0.05,0.02,2.0,2.689714,0.988720,0.275423,1.722713,1.857199
1,Alg1,0.05,0.04,2.0,2.781532,0.991588,0.309315,1.543077,1.682154
2,Alg1,0.05,0.06,2.0,2.874381,0.993732,0.346511,1.370548,1.514267
3,Alg1,0.05,0.08,2.0,2.968226,0.995335,0.377726,1.242023,1.390434
4,Alg1,0.05,0.10,2.0,3.063329,0.996536,0.420390,1.084903,1.238070
...,...,...,...,...,...,...,...,...,...
195,ROAR,0.30,0.42,2.0,5.657109,0.986300,0.796344,0.666573,2.363706
196,ROAR,0.30,0.44,2.0,5.946058,0.981352,0.817293,0.786776,2.570593
197,ROAR,0.30,0.46,2.0,6.185766,0.981376,0.832455,0.762679,2.618409
198,ROAR,0.30,0.48,2.0,6.400879,0.981764,0.851072,0.736165,2.656429


In [290]:
# mask = df_results_avg['algorithm'] == "Alg1"
# df_results_avg.loc[mask, 'algorithm'] = 'Alg1_Phone'
# mask = df_results_avg['algorithm'] == "ROARLInf"
# df_results_avg.loc[mask, 'algorithm'] = 'ROARLInf_Phone'
# df_results_avg

In [291]:
# alphas = [0.02,0.1,0.2]
# lambdas = [0.1,0.2]
# # alphas = np.arange(0.02, 0.52, 0.02).round(2)
# # lambdas = [0.05, 0.1, 0.2, 0.3]
# maxAlpha = max(alphas)

# params = {}
# # 'lr', 'nn'
# params['base_model'] = 'lr'
# # 'synthetic', 'german', 'sba'
# params['data'] = 'sba'
# params['seeds'] = range(5)
# # TODO: add your method here, the method name should match the name in filepath
# # params['algorithms'] = ['Alg1', 'L1PSD', 'ROARLInf', 'ROARL1']
# params['algorithms'] = ['Alg1', 'ROAR']
# # 'one', 'many'
# params['adv_method'] = 'one'

# results = {
#     'algorithm': [],
#     'seed': [],
#     'alpha': [],
#     'lambda': [],
#     'Cost': [],
#     'Current Validity': [],
#     'Worst Case Validity': [],
#     'BCE Loss': [],
#     'J': []
# }

# for algorithm in params['algorithms']:
#     for seed in params['seeds']:
#         for v_alpha in alphas:
#             for v_lamb in lambdas:
#                 # data = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")
#                 data = pd.read_pickle(f"../results/recourse-2025_08/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")
#                 alpha = data["alpha"].unique().item()
#                 lamb = data["lambda"].unique().item()
#                 theta_0 = data["theta_0"].iloc[0]
#                 weights_0, bias_0, = theta_0[:-1], theta_0[[-1]]
#                 X_0 = np.stack(data["x_0"])
#                 X_r = np.stack(data["x_r"])
#                 if params['adv_method'] == 'one':
#                     res = evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
#                 elif params['adv_method'] == 'alpha':
#                     res = evaluate_performance_largestAlpha(X_0, X_r, theta_0, alpha, maxAlpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
#                 else:
#                     res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
#                 results['algorithm'].append(algorithm)
#                 results['seed'].append(seed)
#                 results['alpha'].append(alpha)
#                 results['lambda'].append(lamb)
#                 results['Cost'].append(res['cost'])
#                 results['Current Validity'].append(res['m1_probability'])
#                 results['Worst Case Validity'].append(res['wc_probability'])
#                 results['BCE Loss'].append(res['loss'])
#                 results['J'].append(res['J'])

# df_results = pd.DataFrame(results)

# df_results_avg2 = df_results.groupby(['algorithm', 'lambda', 'alpha'], as_index=False).mean()
# df_results_avg2

In [292]:
# mask = df_results_avg2['algorithm'] == "Alg1"
# df_results_avg2.loc[mask, 'algorithm'] = 'Alg1_Kshitij'
# mask = df_results_avg2['algorithm'] == "ROAR"
# df_results_avg2.loc[mask, 'algorithm'] = 'ROARLInf_Kshitij'

# # df_results_avg2[df_results_avg2['algorithm'] == "Alg1"] = "Alg1_Kshitij"
# # df_results_avg2[df_results_avg2['algorithm'] == "ROAR"] = "ROARLInf_Kshitij"

# tmp_lamb = 0.1
# df_compare = pd.concat([df_results_avg, df_results_avg2])
# df_compare = df_compare[df_compare['lambda'] == tmp_lamb]
# px.line(df_compare, x="alpha", y="J", color="algorithm",
#         title=f"{params['base_model']}_{params['data']}_lamb{tmp_lamb}",
#            markers=True)

In [305]:
tmp_lamb = 0.1

df_results_tmp = df_results_avg[df_results_avg['lambda'] == tmp_lamb]
df_results_tmp
px.line(df_results_tmp, x="alpha", y="J", color="algorithm",
        title=f"{params['base_model']}_{params['data']}_alpha{tmp_lamb}",
        markers=True)

In [293]:
data_map = {'synthetic': 'Synthetic', 'sba': 'Small Business Administration', 'german': 'German', 'income': 'ACS Income'}
model_map = {'lr': 'Logistic Regression', 'nn': 'Neural Network'}

# colors = ['#1f77b4', '#17becf', '#9467bd', '#e377c2', '#2ca02c'] # Synthesis
colors = ['#C7E8F0', "#7FCBDC", "#37AEC8", '#236F80', '#E2C2F4', "#BD74E7", "#9726D9", "#61188B"] # Synthesis
# colors = ['#17becf', '#e377c2', '#2ca02c'] # German
# colors = ['#17becf', '#9467bd', '#e377c2', '#2ca02c'] # SBA

nc = len(colors)
font_family = 'Times New Roman'
font_color = 'black'
width, height = 720, 540

symbols = ['x' for _ in range(len(lambdas))] + ['circle']
size = [7 for _ in range(len(lambdas))] + [5]

fig = go.Figure()
c = 0
for i, alg in enumerate(params["algorithms"]):
    for lamb in lambdas:
        df_alg = df_results_avg.copy()
        df_alg = df_alg[(df_alg['algorithm']==alg) & (df_alg['lambda']==lamb)]
        # df_alg = df_results[(df_results['algorithm'] == alg) & (df_results['lambda']==lamb) & (df_results['alpha']<=0.2)].sort_values(['Cost'], ascending=True).copy().reset_index(drop=True)
        # x, y, mask = find_pareto(df_alg['Cost'], df_alg['Worst Case Validity'], return_index=True)
        # df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(λ={lamb})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha'][mask]})
        x, y = df_alg['Cost'], df_alg['Worst Case Validity']
        df_alg = pd.DataFrame({'Algorithm': [f"{alg}_(λ={lamb})" for _ in range(len(x))], 'Cost': x, 'Worst Case Validity': y, 'alpha': df_alg['alpha']})

        fig.add_trace(go.Scatter(
            x = df_alg['Cost'],
            y = df_alg['Worst Case Validity'],
            marker = dict(color=colors[c], size=3),
            mode = 'lines+markers' if alg != 'wachter' else 'markers',
            name = f"{alg} (λ={lamb})",
            showlegend=True,
            customdata=df_alg['alpha'],
            hovertemplate='Cost: %{x}<br>Validity: %{y}<br>alpha: %{customdata}'
        ))
        c+=1

fig.update_xaxes(
    title=dict(
        text='Cost',
        font=dict(
            family=font_family,
            color=font_color,
            size=25
        )
        ), 
    showline=True, 
    mirror=True,
    linecolor='black', 
    gridcolor='lightgrey', 
    zerolinewidth=1,
    zerolinecolor='lightgrey',
    )


fig.update_yaxes(
    title=dict(
        text='Worst Case Validity',
        font=dict(
            family=font_family,
            color=font_color,
            size=25
        ), 
        ), 
    showline=True, 
    mirror=True,
    linecolor='black', 
    gridcolor='lightgrey',
    zerolinewidth=1,
    zerolinecolor='lightgrey',
    )


fig.update_layout(
    width=width,
    height=height,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(t=50,b=25,l=25,r=25),
    title =dict(
        # text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | Average Adversary", 
        text=f"{params['data'].capitalize()} | {params['base_model'].upper()} | WC Adversary", 
        x= 0.5, 
        font=dict(family=font_family, size=20)
        ),
    legend=dict(
        x=0.975, 
        y=0.025, 
        orientation='v',
        xanchor='right',
        font=dict(
            family=font_family,
            color=font_color,
            size=15
            ), 
        bgcolor='rgba(255, 255, 255, 0.7)',
        bordercolor='lightgrey',
        borderwidth=1,
        entrywidth=100.5,
        ),
    xaxis=dict(
        tickfont=dict(
            family=font_family,
            color=font_color,
            size=20,
        ),
    ),
    yaxis=dict(
        tickfont=dict(
            family=font_family,
            color=font_color,
            size=20
        ),
        range=[-0.1,1.1],
    )
)

print(f'{params["data"]}  |  {params["base_model"].upper()}')
fig.show()

sba  |  LR


In [ ]:
# fig.write_html(f"{params['base_model']}_{params['data']}_costValidityTradeoff_{params['adv_method']}Adv.html")